# Predicting the Electronic Band Gap of Inorganic Crystalline Materials from Composition and Structure
### Stage 6 — Train/Test Split, Baselines and Machine-Learning Models

Uses the two feature tables from Stage 5 (`composition_only` and `composition_structure`, 153,597 rows each). This notebook:

1. Builds the **formula-grouped** train/test split (and a plain random split, for a documented comparison)
2. Fits a dummy baseline, a Ridge baseline, and three tree-ensemble models on **both** feature sets
3. Runs grouped cross-validation on the training portion only
4. Evaluates each fitted model on the held-out test set **once**, with no tuning against it

**Protocol:** model choices are judged on cross-validation, not on the test set. Hyperparameters are fixed, sensible values and are *not* tuned here. The test set is never used to change anything.

**No interpretation happens in this notebook.** Comparing models, discussing overfitting and drawing chemical conclusions belong to Stages 7-9.


## Settings

`FAST_MODE = True` uses 3 CV folds and 50 trees, for a quick trial that checks the notebook runs end to end. **Report results only from a full run (`FAST_MODE = False`).** The tree ensembles (Random Forest, Extra Trees) are the slowest part on a free Colab CPU, so the full run may take a while. Keep the tab open, since a disconnected runtime loses in-memory results.

**If the runtime disconnects:** reconnect and choose Runtime > Run all. Finished models are saved to Drive as they complete and are loaded back instead of being recomputed (see the Checkpointing section).


In [ ]:
FAST_MODE = False

RANDOM_SEED = 42
CV_FOLDS = 3 if FAST_MODE else 5
N_TREES = 50 if FAST_MODE else 100
TEST_SIZE = 0.20

print(f"FAST_MODE={FAST_MODE} | CV folds={CV_FOLDS} | trees={N_TREES} | seed={RANDOM_SEED}")


In [ ]:
import os
import json
import time
import shutil
import warnings

import numpy as np
import pandas as pd
import joblib
import sklearn

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error

np.random.seed(RANDOM_SEED)

print("scikit-learn:", sklearn.__version__)
print("pandas:      ", pd.__version__)
print("numpy:       ", np.__version__)


## Checkpointing (resume after a disconnect)

Colab runtimes can disconnect during a long run, and a disconnect wipes everything held in memory. Every finished model (its CV result, its final fit, its test predictions) is therefore saved to a checkpoint folder on Google Drive as soon as it completes. **If the runtime disconnects, reconnect and choose Runtime > Run all. Finished models are loaded from disk instead of being recomputed.**

The checkpoint folder name includes the run settings (fast mode, folds, trees, seed), so a quick `FAST_MODE` trial can never be mistaken for a full run. The split is deterministic (fixed seed on the same data), so a resumed run uses exactly the same train/test split.


In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/materials_bandgap_project"
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

run_tag = f"fast{int(FAST_MODE)}_cv{CV_FOLDS}_trees{N_TREES}_seed{RANDOM_SEED}"
CKPT_DIR = os.path.join(DRIVE_DIR if ON_COLAB else ".", "stage6_checkpoints", run_tag)
os.makedirs(CKPT_DIR, exist_ok=True)

def slug(s):
    return s.replace(" ", "_").replace("(", "").replace(")", "")

print("Checkpoint directory:", CKPT_DIR)
print("Checkpoint files already present:", len(os.listdir(CKPT_DIR)))


## Load the Stage 5 feature tables

Same local-then-Drive fallback as before. One detail matters here: the file is read with `keep_default_na=False` so that the formula `NaN` (sodium nitride) is not turned into a missing value again by pandas. Only truly empty fields are treated as missing.


In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/materials_bandgap_project"

def ensure_local(name):
    if os.path.exists(name):
        return
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass
    drive_path = os.path.join(DRIVE_DIR, name)
    if os.path.exists(drive_path):
        shutil.copy(drive_path, name)
        print(f"Loaded {name} from Google Drive.")
    else:
        raise FileNotFoundError(
            f"{name} not found locally or at {drive_path}. Run Stage 5 first and make sure "
            "its outputs were copied to that Drive folder."
        )

def read_features(name):
    ensure_local(name)
    return pd.read_csv(name, keep_default_na=False, na_values=[""])

comp_only = read_features("mp_bandgap_features_composition_only.csv")
comp_struct = read_features("mp_bandgap_features_composition_structure.csv")

assert comp_only.shape[0] == comp_struct.shape[0], "Row counts differ between feature tables."
assert comp_only["material_id"].equals(comp_struct["material_id"]), "Row order differs between feature tables."
assert comp_struct["formula_pretty"].notna().all(), "Missing formula_pretty values -- check the CSV read."

print("Composition-only table:      ", comp_only.shape)
print("Composition+structure table: ", comp_struct.shape)


## Feature sets and the leakage guard

Identifier/target columns and `energy_above_hull` / `is_stable` are kept out of the model inputs, as decided in the Stage 2 leakage review. The assertions below fail loudly if any excluded column slips into a feature list.


In [ ]:
ID_COLS = ["material_id", "formula_pretty", "band_gap", "energy_above_hull", "is_stable"]

COMP_FEATURES = [c for c in comp_only.columns if c not in ID_COLS]
STRUCT_FEATURES = [c for c in comp_struct.columns if c not in ID_COLS and c not in COMP_FEATURES]

FEATURE_SETS = {
    "composition_only": COMP_FEATURES,
    "composition_structure": COMP_FEATURES + STRUCT_FEATURES,
}

FORBIDDEN = {"band_gap", "cbm", "vbm", "efermi", "is_metal", "is_gap_direct",
             "energy_above_hull", "is_stable", "formation_energy_per_atom", "space_group_number"}
for fs_name, cols in FEATURE_SETS.items():
    assert not (set(cols) & FORBIDDEN), f"Forbidden column in feature set {fs_name}"

print(f"Composition features ({len(COMP_FEATURES)}): {COMP_FEATURES}")
print(f"\nStructure features ({len(STRUCT_FEATURES)}): {STRUCT_FEATURES}")

data = comp_struct
y = data["band_gap"].to_numpy(dtype=float)
groups = data["formula_pretty"].to_numpy()

X_np = {fs: data[cols].astype(float).to_numpy() for fs, cols in FEATURE_SETS.items()}


## Train/test split

**Primary split: grouped by formula.** All polymorphs of one formula land on the same side of the split. Otherwise the model could be trained on one polymorph of a compound and tested on another with the same composition features, which would overstate how well it generalizes to new chemistry. Stage 4 found that about 20% of formulas have more than one polymorph, so this matters here.

**Comparison split: plain random.** Built only so we can measure, later in this notebook, how much a naive split overstates performance.

**Limitation to state honestly:** grouping by formula prevents identical-composition leakage, but it does not prevent *chemically similar* compounds (for example, related compositions in the same chemical system) from appearing on both sides. A stricter grouping by chemical system would be a useful robustness check to note under future work.


In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(data, y, groups))

assert set(groups[train_idx]).isdisjoint(set(groups[test_idx])), "Formula overlap between grouped train and test!"

print("GROUPED split")
print(f"  train rows: {len(train_idx)} | test rows: {len(test_idx)} ({len(test_idx)/len(data):.1%})")
print(f"  train formulas: {len(set(groups[train_idx]))} | test formulas: {len(set(groups[test_idx]))}")
print("  formula overlap between train and test: 0 (asserted)")

rand_train_idx, rand_test_idx = train_test_split(
    np.arange(len(data)), test_size=TEST_SIZE, random_state=RANDOM_SEED
)
train_formulas_rand = set(groups[rand_train_idx])
overlap_frac = np.mean([g in train_formulas_rand for g in groups[rand_test_idx]])

print("\nRANDOM split (comparison only)")
print(f"  train rows: {len(rand_train_idx)} | test rows: {len(rand_test_idx)}")
print(f"  share of random-test rows whose formula also appears in random-train: {overlap_frac:.1%}")

split_df = pd.DataFrame({"material_id": data["material_id"], "grouped_split": "train", "random_split": "train"})
split_df.loc[test_idx, "grouped_split"] = "test"
split_df.loc[rand_test_idx, "random_split"] = "test"
split_df.to_csv("mp_bandgap_stage6_split_ids.csv", index=False)


## Missing values and how each model handles them

26 rows have `NaN` in a composition feature (noble-gas-only compounds, see Stage 5). Nothing is imputed:

- `HistGradientBoostingRegressor` accepts `NaN` natively, so it trains on **all** training rows.
- Ridge, Random Forest, Extra Trees and the dummy baseline are fitted on **complete-case** training rows only.
- **Every model is evaluated on the same complete-case rows**, so all reported metrics use an identical set of validation/test rows and the comparison stays fair.


In [ ]:
all_cols = FEATURE_SETS["composition_structure"]
complete = ~np.isnan(X_np["composition_structure"]).any(axis=1)

print(f"Rows with any missing feature value: {(~complete).sum()} of {len(complete)}")
print(f"  in grouped train: {(~complete[train_idx]).sum()} | in grouped test: {(~complete[test_idx]).sum()}")


## Models

| Model | Why it is included |
|---|---|
| Dummy (mean) | The floor: predicts the training mean for everything. Any real model must beat it. |
| Ridge regression | Simple, interpretable linear baseline (features standardised first). Shows how much a purely linear picture explains. |
| Random Forest | Robust, low-tuning tree ensemble that handles nonlinearity and feature interactions. |
| Extra Trees | Similar to Random Forest but with more randomised splits, giving a different bias-variance trade-off and often faster fits. |
| HistGradientBoosting | Boosted trees, efficient on large tabular data, and handles missing values natively. |

All hyperparameters are fixed, standard values. **They are not tuned in this stage.** Tuning, if done, will use cross-validation only and be reported as such.


In [ ]:
MODEL_FACTORIES = {
    "Dummy (mean)": lambda: DummyRegressor(strategy="mean"),
    "Ridge": lambda: make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "RandomForest": lambda: RandomForestRegressor(
        n_estimators=N_TREES, min_samples_leaf=3, n_jobs=-1, random_state=RANDOM_SEED),
    "ExtraTrees": lambda: ExtraTreesRegressor(
        n_estimators=N_TREES, min_samples_leaf=3, n_jobs=-1, random_state=RANDOM_SEED),
    "HistGradientBoosting": lambda: HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.1, early_stopping=False, random_state=RANDOM_SEED),
}
NAN_OK = {"Dummy (mean)": False, "Ridge": False, "RandomForest": False,
          "ExtraTrees": False, "HistGradientBoosting": True}

def metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
        "MedAE": float(median_absolute_error(y_true, y_pred)),
    }

def fit_model(model_name, X, fit_idx):
    idx = fit_idx if NAN_OK[model_name] else fit_idx[complete[fit_idx]]
    model = MODEL_FACTORIES[model_name]()
    t0 = time.time()
    model.fit(X[idx], y[idx])
    return model, time.time() - t0, len(idx)

def predict_on_complete(model, X, eval_idx):
    idx = eval_idx[complete[eval_idx]]
    return idx, model.predict(X[idx])

print("Models defined:", list(MODEL_FACTORIES))


## Cross-validation (training portion only)

`GroupKFold` keeps every formula's polymorphs inside a single fold, so the validation folds are subject to the same no-formula-overlap rule as the final test set. The held-out test set is not touched in this cell.


In [ ]:
gkf = GroupKFold(n_splits=CV_FOLDS)
cv_rows = []

t_start = time.time()
for fs_name in FEATURE_SETS:
    X = X_np[fs_name]
    for model_name in MODEL_FACTORIES:
        ck = os.path.join(CKPT_DIR, f"cv_{fs_name}_{slug(model_name)}.json")
        if os.path.exists(ck):
            with open(ck) as f:
                row = json.load(f)
            cv_rows.append(row)
            print(f"[resumed from checkpoint] {fs_name:22s} {model_name:22s} CV MAE={row['CV_MAE']:.4f}")
            continue

        fold_metrics = []
        for k, (tr_rel, va_rel) in enumerate(gkf.split(train_idx, y[train_idx], groups[train_idx])):
            ck_fold = os.path.join(CKPT_DIR, f"cvfold_{fs_name}_{slug(model_name)}_{k}.json")
            if os.path.exists(ck_fold):
                with open(ck_fold) as f:
                    fold_metrics.append(json.load(f))
                continue
            tr, va = train_idx[tr_rel], train_idx[va_rel]
            assert set(groups[tr]).isdisjoint(set(groups[va]))
            model, _, _ = fit_model(model_name, X, tr)
            va_idx, va_pred = predict_on_complete(model, X, va)
            fold_m = metrics(y[va_idx], va_pred)
            with open(ck_fold, "w") as f:
                json.dump(fold_m, f)
            fold_metrics.append(fold_m)
            print(f"    [{time.time()-t_start:7.0f}s] {fs_name} / {model_name}: fold {k+1}/{CV_FOLDS} done")
        fm = pd.DataFrame(fold_metrics)
        row = {
            "feature_set": fs_name, "model": model_name,
            **{f"CV_{k}": float(fm[k].mean()) for k in ["MAE", "RMSE", "R2", "MedAE"]},
            **{f"CV_{k}_std": float(fm[k].std()) for k in ["MAE", "RMSE", "R2"]},
        }
        with open(ck, "w") as f:
            json.dump(row, f)
        cv_rows.append(row)
        print(f"[{time.time()-t_start:7.0f}s] {fs_name:22s} {model_name:22s} "
              f"CV MAE={row['CV_MAE']:.4f}  RMSE={row['CV_RMSE']:.4f}  R2={row['CV_R2']:.4f}")

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv("stage6_cv_results.csv", index=False)


## Final fit on the full training portion and one-time test evaluation

Each model is refitted on the entire training portion, then scored on (a) the training rows and (b) the held-out test rows. Reporting train and test metrics side by side is what lets Stage 7 discuss overfitting. Fit time is recorded as a rough measure of computational cost. Nothing in this cell feeds back into any model choice.


In [ ]:
final_rows = []
pred_frames = []
fitted_models = {}

for fs_name in FEATURE_SETS:
    X = X_np[fs_name]
    for model_name in MODEL_FACTORIES:
        ck_row = os.path.join(CKPT_DIR, f"final_{fs_name}_{slug(model_name)}.json")
        ck_pred = os.path.join(CKPT_DIR, f"pred_{fs_name}_{slug(model_name)}.csv")
        ck_model = os.path.join(CKPT_DIR, f"hgb_{fs_name}.joblib")
        is_hgb = model_name == "HistGradientBoosting"

        if os.path.exists(ck_row) and os.path.exists(ck_pred) and (not is_hgb or os.path.exists(ck_model)):
            with open(ck_row) as f:
                final_rows.append(json.load(f))
            pred_frames.append(pd.read_csv(ck_pred, keep_default_na=False, na_values=[""]))
            if is_hgb:
                fitted_models[fs_name] = joblib.load(ck_model)
            print(f"[resumed from checkpoint] {fs_name:22s} {model_name:22s}")
            continue

        model, fit_sec, n_fit = fit_model(model_name, X, train_idx)

        tr_idx_c, tr_pred = predict_on_complete(model, X, train_idx)
        te_idx_c, te_pred = predict_on_complete(model, X, test_idx)
        m_tr, m_te = metrics(y[tr_idx_c], tr_pred), metrics(y[te_idx_c], te_pred)

        row = {
            "feature_set": fs_name, "model": model_name,
            "n_fit_rows": int(n_fit), "n_test_rows_evaluated": int(len(te_idx_c)),
            "fit_seconds": round(fit_sec, 1),
            **{f"Train_{k}": v for k, v in m_tr.items()},
            **{f"Test_{k}": v for k, v in m_te.items()},
        }
        pframe = pd.DataFrame({
            "material_id": data["material_id"].to_numpy()[te_idx_c],
            "formula_pretty": groups[te_idx_c],
            "feature_set": fs_name, "model": model_name,
            "y_true": y[te_idx_c], "y_pred": te_pred,
        })

        if is_hgb:
            joblib.dump(model, ck_model)
            fitted_models[fs_name] = model
        pframe.to_csv(ck_pred, index=False)
        with open(ck_row, "w") as f:
            json.dump(row, f)

        final_rows.append(row)
        pred_frames.append(pframe)
        print(f"{fs_name:22s} {model_name:22s} fit {fit_sec:6.1f}s | "
              f"Train MAE={m_tr['MAE']:.4f} | Test MAE={m_te['MAE']:.4f}")

final_df = pd.DataFrame(final_rows)
pred_df = pd.concat(pred_frames, ignore_index=True)


## Results table

Only genuinely computed values appear here. Cross-validation columns come from the grouped CV on the training portion, and train/test columns come from the final fits. Rows are ordered by feature set, then by CV MAE. **Model selection is judged on CV, not test.**


In [ ]:
results = cv_df.merge(final_df, on=["feature_set", "model"])
results = results.sort_values(["feature_set", "CV_MAE"]).reset_index(drop=True)

display_cols = ["feature_set", "model", "CV_MAE", "CV_RMSE", "CV_R2",
                "Train_MAE", "Train_R2", "Test_MAE", "Test_RMSE", "Test_R2", "fit_seconds"]
pd.set_option("display.width", 200)
print(results[display_cols].round(4).to_string(index=False))

results.to_csv("stage6_model_results.csv", index=False)


## Documented comparison: random split vs grouped split

One model (HistGradientBoosting on composition+structure) is refitted on the random split and scored on its random test set, then compared with the same model's grouped-split test result above. This quantifies how much a naive random split flatters the model when polymorphs of the same formula are spread across train and test. The two test sets contain different rows, so treat it as an illustration of the split effect and not a controlled paired comparison.


In [ ]:
fs_name, model_name = "composition_structure", "HistGradientBoosting"
X = X_np[fs_name]

model_r, _, _ = fit_model(model_name, X, rand_train_idx)
idx_r, pred_r = predict_on_complete(model_r, X, rand_test_idx)
m_random = metrics(y[idx_r], pred_r)

m_grouped = final_df[(final_df.feature_set == fs_name) & (final_df.model == model_name)].iloc[0]

comparison = pd.DataFrame({
    "split": ["random", "grouped by formula"],
    "test_MAE": [m_random["MAE"], m_grouped["Test_MAE"]],
    "test_RMSE": [m_random["RMSE"], m_grouped["Test_RMSE"]],
    "test_R2": [m_random["R2"], m_grouped["Test_R2"]],
})
print(f"Share of random-test rows whose formula also appears in random-train: {overlap_frac:.1%}\n")
print(comparison.round(4).to_string(index=False))
comparison.to_csv("stage6_random_vs_grouped.csv", index=False)


## Save outputs

Large ensemble models are **not** saved (they can reach hundreds of MB). Stage 8 can refit whichever model is chosen for explainability. The small HistGradientBoosting models, the split ids, the results tables, the test-set predictions (for Stage 8 error analysis) and the run configuration are saved and copied to Drive.


In [ ]:
for fs_name, model in fitted_models.items():
    joblib.dump(model, f"stage6_hgb_{fs_name}.joblib")

pred_df.to_csv("stage6_test_predictions.csv", index=False)

config = {
    "random_seed": RANDOM_SEED, "fast_mode": FAST_MODE, "cv_folds": CV_FOLDS,
    "n_trees": N_TREES, "test_size": TEST_SIZE,
    "sklearn_version": sklearn.__version__, "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "split": "GroupShuffleSplit on formula_pretty (grouped); train_test_split (random, comparison only)",
    "n_train_rows": int(len(train_idx)), "n_test_rows": int(len(test_idx)),
    "n_rows_with_missing_features": int((~complete).sum()),
    "random_split_formula_overlap_fraction": float(overlap_frac),
    "hyperparameters": "fixed, untuned; see MODEL_FACTORIES in the notebook",
}
with open("stage6_config.json", "w") as f:
    json.dump(config, f, indent=2)

OUT_FILES = [
    "mp_bandgap_stage6_split_ids.csv", "stage6_cv_results.csv", "stage6_model_results.csv",
    "stage6_random_vs_grouped.csv", "stage6_test_predictions.csv", "stage6_config.json",
] + [f"stage6_hgb_{fs}.joblib" for fs in fitted_models]

try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for fname in OUT_FILES:
        if os.path.exists(fname):
            shutil.copy(fname, os.path.join(DRIVE_DIR, fname))
            print(f"Copied {fname} to Drive")
        else:
            print(f"WARNING: {fname} not found")
except ImportError:
    print("Not running in Colab -- files remain in the local working directory.")


### Stage 6 ends here.

Models are built, cross-validated and scored once on the held-out test set. **No comparison, interpretation or chemical discussion has been done.** That is Stage 7.

**Do not run Stage 7 code in this notebook.** Await confirmation before proceeding.
